## Setup, Imports, and Triton Fixes

In [1]:
!pip install -q --no-index --find-links /kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages datasets trl --ignore-installed

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.21.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
torchvision 0.26.0.dev20260315+cu128 requires torch==2.12.0.dev20260315, but you have torch 2.10.0 which is incompatible.
ydata-profiling 4.18.1 requires pandas!=1.4.0,<3.0,>1.5, but you have pandas 3.0.1 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.1 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
db-dtypes 1.5.0 requires 

In [2]:
import datasets
import trl
print("Successfully imported datasets version:", datasets.__version__)
print("Successfully imported trl version:", trl.__version__)

Successfully imported datasets version: 4.3.0
Successfully imported trl version: 0.24.0


In [3]:
import multiprocessing

multiprocessing.cpu_count()

48

In [4]:
import os
import sys
import stat
import shutil
import gc
import zipfile
import polars as pl
import torch
import torch.nn.functional as F
import kagglehub
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

# --- Kaggle / Triton Environment Fixes ---
# Bypass Triton rmsnorm (avoids ptxas-blackwell permission issue)
def _pure_rmsnorm_fn(x, weight, bias=None, z=None, eps=1e-5,
                     group_size=None, norm_before_gate=True, upcast=True):
    dtype = x.dtype
    if upcast:
        x = x.float()
    variance = x.pow(2).mean(-1, keepdim=True)
    x_normed = x * torch.rsqrt(variance + eps)
    out = x_normed * weight.float()
    if bias is not None:
        out = out + bias.float()
    if z is not None:
        out = out * F.silu(z.float())
    return out.to(dtype)

for name, mod in list(sys.modules.items()):
    if hasattr(mod, 'rmsnorm_fn'):
        mod.rmsnorm_fn = _pure_rmsnorm_fn

# Copy PTXAS binaries to a writable temp directory
src = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell"
dst = "/tmp/ptxas-blackwell"
if os.path.exists(src):
    shutil.copy2(src, dst)
    os.chmod(dst, os.stat(dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

    import triton.backends.nvidia as nv_backend
    src_bin = os.path.join(os.path.dirname(nv_backend.__file__), "bin")
    dst_bin = "/tmp/triton_nvidia_bin"
    shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
    for f in os.listdir(dst_bin):
        fp = os.path.join(dst_bin, f)
        if os.path.isfile(fp):
            os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

    nv_backend.__file__ = os.path.join(dst_bin, "..", "__init__.py")
    os.environ["TRITON_PTXAS_PATH"] = dst

# --- Hyperparameters ---
SAMPLES_PER_TYPE = 100   # 每种题型抽 100 条, 6 种 = 600 条
LORA_RANK = 32
MAX_SEQ_LEN = 1024
NUM_EPOCHS = 1
GRAD_ACCUM = 4           # 有效 batch_size = 4
LR = 2e-4

OUTPUT_DIR = "/kaggle/working/adapter"
os.makedirs(OUTPUT_DIR, exist_ok=True)

/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/torch/compiler/__init__.py:148: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  return torch._dynamo.allow_in_graph(fn)


## Data Loading & Formatting

In [5]:
# Download model
MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")

# Load data
train_df = pl.read_csv('/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv')
print(f"Total training samples available: {len(train_df)}")

# --- 按题型分层抽样 ---
def classify_type(prompt_text):
    p = prompt_text.lower()
    if 'bit manipulation' in p or '8-bit binary' in p: return 'bit_ops'
    elif 'encrypt' in p or 'decrypt' in p: return 'cipher'
    elif 'gravitational' in p or 'falling distance' in p: return 'gravity'
    elif 'numeral system' in p: return 'numeral'
    elif 'transformation rules' in p: return 'symbol'
    elif 'unit conversion' in p or 'convert the following measurement' in p: return 'unit_conv'
    return 'unknown'

train_df = train_df.with_columns(
    pl.col("prompt").map_elements(classify_type, return_dtype=pl.Utf8).alias("qtype")
)

# 每种题型抽 SAMPLES_PER_TYPE 条
sampled_dfs = []
for qtype in train_df["qtype"].unique().to_list():
    subset = train_df.filter(pl.col("qtype") == qtype)
    n = min(SAMPLES_PER_TYPE, len(subset))
    sampled_dfs.append(subset.sample(n=n, seed=42))
    print(f"  {qtype}: {len(subset)} available, sampled {n}")

train_df = pl.concat(sampled_dfs).drop("qtype")
print(f"Total sampled: {len(train_df)}")

# Convert to Hugging Face Dataset
hf_dataset = Dataset.from_pandas(train_df.to_pandas())

# Initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 评测代码固定用这个后缀, 训练必须匹配
METRIC_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

def build_training_text(example):
    prompt = example["prompt"]
    answer = str(example["answer"])

    # 用评测的精确 prompt 格式
    user_msg = prompt + METRIC_SUFFIX

    # 简洁格式: 直接给答案, 不加假 CoT
    assistant_msg = f"\\boxed{{{answer}}}"

    messages = [
        {"role": "user", "content": user_msg},
        {"role": "assistant", "content": assistant_msg},
    ]

    for kwargs in [{"enable_thinking": True}, {}]:
        try:
            text = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=False, **kwargs
            )
            return {"text": text}
        except Exception:
            continue

    # fallback
    text = (
        f"<|im_start|>user\n{user_msg}<|im_end|>\n"
        f"<|im_start|>assistant\n{assistant_msg}<|im_end|>"
    )
    return {"text": text}

Total training samples available: 9500
  numeral: 1576 available, sampled 100
  gravity: 1597 available, sampled 100
  cipher: 1576 available, sampled 100
  unit_conv: 1594 available, sampled 100
  symbol: 1555 available, sampled 100
  bit_ops: 1602 available, sampled 100
Total sampled: 600


In [6]:
hf_dataset = hf_dataset.map(
    build_training_text,
    remove_columns=hf_dataset.column_names
)

# 显示样本信息
print(f"Dataset size: {len(hf_dataset)}")
print(f"\nExample (first 800 chars):\n{hf_dataset[0]['text'][:800]}")
print(f"\n--- Token length stats ---")
lengths = [len(tokenizer.encode(t['text'])) for t in hf_dataset.select(range(min(100, len(hf_dataset))))]
print(f"  Sample of 100: min={min(lengths)}, max={max(lengths)}, avg={sum(lengths)//len(lengths)}")
print(f"  MAX_SEQ_LEN={MAX_SEQ_LEN}, truncation will apply to sequences > {MAX_SEQ_LEN}")

Map:   0%|          | 0/600 [00:00<?, ? examples/s]

Dataset size: 600

Example (first 800 chars):
<|im_start|>system
<|im_end|>
<|im_start|>user
In Alice's Wonderland, numbers are secretly converted into a different numeral system. Some examples are given below:
45 -> XLV
15 -> XV
17 -> XVII
Now, write the number 92 in the Wonderland numeral system.
Please put your final answer inside `\boxed{}`. For example: `\boxed{your answer}`<|im_end|>
<|im_start|>assistant
<think></think>\boxed{XCII}<|im_end|>


--- Token length stats ---
  Sample of 100: min=96, max=115, avg=105
  MAX_SEQ_LEN=1024, truncation will apply to sequences > 1024


## Model Loading & LoRA Setup

In [7]:
# Load Model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, 
    device_map="auto", 
    trust_remote_code=True, 
    dtype=torch.bfloat16
)
print(f"Model loaded. Vocab size: {len(tokenizer)}")

# Force slow path — bypass the broken CUDA kernels
for name, mod in sys.modules.items():
    if "modeling_nemotron_h" in name:
        mod.is_fast_path_available = False
        print(f"Patched {name}: is_fast_path_available = False")

# 恢复 all-linear: 覆盖更多参数, 学习能力更强
# 0.66 版本就是 all-linear + alpha=16, vLLM 加载无问题
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=16,         # 恢复原值, alpha/rank=0.5 更稳定
    target_modules="all-linear",
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/6243 [00:00<?, ?it/s]

Model loaded. Vocab size: 131072
Patched transformers_modules._1.modeling_nemotron_h: is_fast_path_available = False


/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_tensor.py:122: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_tensor.py:195: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_scaling_utils.py:90: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_linear.py:60: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_

trainable params: 883,873,792 || all params: 32,461,811,136 || trainable%: 2.7228


## Training with SFTTrainer

In [8]:
import os
import triton.backends.nvidia.compiler as nv_compiler

# Tell Triton's environment parser where the writable Blackwell binary is
os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = "/tmp/ptxas-blackwell"
nv_compiler.get_ptxas_version = lambda arch: "12.0"

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,  # 有效 batch_size = 4
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LR,
    logging_steps=10,                        # 600/4=150 步, 每 10 步 log
    bf16=True,
    max_grad_norm=1.0,
    optim="adamw_torch",
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    save_strategy="no",
    report_to="none",
    dataset_text_field="text",
    max_length=MAX_SEQ_LEN,
    packing=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": True},
)

trainer = SFTTrainer(
    model=model,
    train_dataset=hf_dataset,
    processing_class=tokenizer,
    args=training_args
)

# 显示训练计划
total_steps = len(hf_dataset) // GRAD_ACCUM
print(f"Training plan: {len(hf_dataset)} samples × {NUM_EPOCHS} epoch")
print(f"  Effective batch size: {GRAD_ACCUM}")
print(f"  Total optimizer steps: ~{total_steps}")
print(f"  Max sequence length: {MAX_SEQ_LEN}")
print(f"  Learning rate: {LR}")
print("Starting training...")
trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/600 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/600 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/600 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 11, 'pad_token_id': 11}.


Training plan: 600 samples × 1 epoch
  Effective batch size: 4
  Total optimizer steps: ~150
  Max sequence length: 1024
  Learning rate: 0.0002
Starting training...


Step,Training Loss
10,11.288831
20,6.641730
30,4.884527
40,3.591575
50,4.626636
60,3.651370
70,2.926567
80,3.374972
90,3.261403
100,3.485331


TrainOutput(global_step=150, training_loss=4.27978193918864, metrics={'train_runtime': 1860.8456, 'train_samples_per_second': 0.322, 'train_steps_per_second': 0.081, 'total_flos': 1.90670644573344e+16, 'train_loss': 4.27978193918864, 'epoch': 1.0})

## Save and Package Submission

In [9]:
# Save adapter weights and config
trainer.model.save_pretrained(OUTPUT_DIR)
print(f"Adapter files saved to {OUTPUT_DIR}:")
for f in os.listdir(OUTPUT_DIR):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f"  {f} ({size/1024:.1f} KB)")

# Package into submission.zip (required by competition)
zip_path = "/kaggle/working/submission.zip"
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(OUTPUT_DIR):
        fpath = os.path.join(OUTPUT_DIR, fname)
        zf.write(fpath, fname)  # files at zip root

print(f"\nCreated {zip_path} ({os.path.getsize(zip_path)/1024/1024:.1f} MB)")

# Verify it contains adapter_config.json (required)
with zipfile.ZipFile(zip_path, 'r') as zf:
    names = zf.namelist()
    print(f"Contents: {names}")
    assert "adapter_config.json" in names, "Missing adapter_config.json!"
    print("✓ submission.zip looks correct and is ready to submit!")

Adapter files saved to /kaggle/working/adapter:
  adapter_model.safetensors (3454393.7 KB)
  adapter_config.json (1.1 KB)
  README.md (5.2 KB)

Created /kaggle/working/submission.zip (3006.3 MB)
Contents: ['adapter_model.safetensors', 'adapter_config.json', 'README.md']
✓ submission.zip looks correct and is ready to submit!
